## Final Project - Linear Modeling 

##### Regression ex. in scikit-learn 
###### week9-1practice_xian.ipynb
- Supervised learning 
- continuous response regression task 

##### Intro to Deep Learning & PyTorch 

###### week11practice.ipynb
- Basic datatype 
- creating tensors 
- operation examples 
- tensor to/from a NumPy array 
- torch.device("cuda") --> tensor on GPU 
- automatic differentiation & gradients 
- perceptron & MLP 
    - forward propogation 
    - [(nonlinear) activation functions](https://docs.pytorch.org/docs/stable/nn.functional.html) (sigmoid, hyperbolic tangent, ReLU) 
    - shallow (single-layer) vs. deep (multi-layer) NN 

###### week11-2.ipynb
- loss function
    - mean square error: continuous regression (response) task  
    - binary cross entropy: binary classification 
- [Training algorithms](https://docs.pytorch.org/docs/stable/optim.html) 
    - [Gradient descent](https://www.geeksforgeeks.org/machine-learning/gradient-descent-algorithm-and-its-variants/): general algorithm; computes gradient for each data point 
    - alternative algorithms 
        - stochastic GD: only one training example is used to compute the gradient and update the parameters at each iteration
            - can be faster than batch gradient descent but may lead to more noise in the updates
        - mini-batch GD: a small batch of training examples is used to compute the gradient and update the parameters at each iteration
            - can be a good compromise; can be faster than GD and less noisy than Stochastic GD
        - Momentum-based GD 
        - Nesterov Accelerated Gradient (NAG) 
        - Adagrad 
        - RMSProp
        - Adam 
    - backpropogation
    - learning rate 
        - small $\alpha$ converges slowly and gets stuck at false local minima 
        - large $\alpha$ overshoots, becomes unstable, and diverges
        - choosing
            - trial-and-error 
            - adaptive learning rate

### Setup

In [ ]:
# load necessary libraries
import os 
import pandas as pd 
from sklearn.model_selection import train_test_split 
import numpy as np
import random
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import torch 
import torch.nn as nn # base class for all neural network modules
from sklearn.preprocessing import StandardScaler
import math
from copy import deepcopy

In [14]:
# set random seed for reproducibility
seed = 42

# set up working directory; user changes as needed
os.getcwd()
os.listdir()
# os.chdir("C:/Users/Bryant Willoughby/OneDrive/Documents/ST507/Project")


['.Rhistory',
 'austin_micromobility.csv',
 'austin_micromobility_metadata.txt',
 'DataManipulation.ipynb',
 'Final_project_guideline.pdf',
 'Heatmap.png',
 'LinearModeling.ipynb',
 'LossMLP.png',
 'NonLinearModeling.ipynb',
 'PerceptronLoss.png',
 'ProjectProposal.pdf',
 'ProjectProposalSubmission.pdf',
 'ProjectScript1.ipynb',
 'ProjectScript2.ipynb',
 'ProjectScript3.ipynb',
 'proposal_guideline.pdf',
 'test_data.csv',
 'train_data.csv',
 'val_data.csv']

### Modeling 

All of the preceding manipulation is completed. Read in the cleaned and formatted training, validation, and testing datasets directly to start modeling. 

In [3]:
train_df = pd.read_csv("train_data.csv")
val_df   = pd.read_csv("val_data.csv")
test_df  = pd.read_csv("test_data.csv")

X_train, y_train = train_df.drop(columns=['_log_trip_distance']), train_df['_log_trip_distance']
X_val, y_val     = val_df.drop(columns=['_log_trip_distance']),   val_df['_log_trip_distance']
X_test, y_test   = test_df.drop(columns=['_log_trip_distance']),  test_df['_log_trip_distance']

scaler = StandardScaler() 
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (1026295, 10) (1026295,)
Validation: (219920, 10) (219920,)
Test: (219921, 10) (219921,)


#### Baseline Linear Regression Model 

This is a multiple linear regression model that cannot adequately capture the complexity of the data. It is a simple basline model with first-order terms that only capture linear relationships between the predictors and response. 

In [4]:
# --- Fit linear regression model ---
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predictions (still log scale)
y_test_pred_log = lr.predict(X_test)

# --- Log-scale evaluation ---
mse_log = mean_squared_error(y_test, y_test_pred_log)
r2_log = r2_score(y_test, y_test_pred_log)

# Intercept first, then predictors in the same column order as X_train
coef_index = ['Intercept'] + X_train.columns.tolist()
coef_values = np.concatenate(([lr.intercept_], lr.coef_))
coef_df = pd.DataFrame({'Feature': coef_index, 'Coefficient': coef_values, 'Magnitude': np.sort(np.abs(coef_values))[::-1]})

print("MSE (log-scale):", mse_log)
print("R2 (log-scale):", r2_log)
print("Feature Coefficients (ordered):")
print(coef_df)


MSE (log-scale): 1.565775578736244
R2 (log-scale): 0.04768757115727562
Feature Coefficients (ordered):
                     Feature   Coefficient     Magnitude
0                  Intercept  6.775750e+00  6.775750e+00
1              Trip Duration  1.705665e-01  1.705665e-01
2                      Month -1.039728e-01  1.247607e-01
3                       Hour  7.194593e-03  1.039728e-01
4                Day of Week  2.724917e-02  8.676075e-02
5   Council District (Start) -5.387077e-02  5.387077e-02
6     Council District (End) -8.676075e-02  2.724917e-02
7          Year (US/Central)  1.032160e-16  1.212390e-02
8                veh_scooter -1.247607e-01  7.194593e-03
9                   start_ts -3.204126e-03  3.204126e-03
10                    end_ts  1.212390e-02  1.032160e-16


#### Preprocessing for PyTorch

In [5]:
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
X_val_t   = torch.tensor(X_val.values, dtype=torch.float32)
X_test_t  = torch.tensor(X_test.values, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1) # reshape to column vector
y_val_t   = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [6]:
print(f"X_train_t.shape: {X_train_t.shape}, y_train_t.shape: {y_train_t.shape}")
print(f"X_val_t.shape: {X_val_t.shape}, y_val_t.shape: {y_val_t.shape}")
print(f"X_test_t.shape: {X_test_t.shape}, y_test_t.shape: {y_test_t.shape}")

X_train_t.shape: torch.Size([1026295, 10]), y_train_t.shape: torch.Size([1026295, 1])
X_val_t.shape: torch.Size([219920, 10]), y_val_t.shape: torch.Size([219920, 1])
X_test_t.shape: torch.Size([219921, 10]), y_test_t.shape: torch.Size([219921, 1])


#### Single-Layer Perceptron 

I first fit a single-layer perceptron model. From its construction, this model (like previous linear regression task) only captures the linear relationship between predictors and the response. However, the architecture and model training procedure are different. 

In [7]:
# Simple 1-layer neural net
class Perceptron(nn.Module): 
    def __init__(self, input_dim): 
        super().__init__() # initialize parent class
        self.linear = nn.Linear(input_dim, 1) # output dim = 1 for regression
    
    def forward(self, x): 
        return self.linear(x)

# --- Training function for single-layer perceptron ---
def train_slp(model, loss_fn, optimizer,
              X_train, y_train,
              X_val, y_val,
              epochs=2500):
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs + 1):
        model.train()
        optimizer.zero_grad() # reset gradients
        y_train_pred = model(X_train)
        train_loss = loss_fn(y_train_pred, y_train)
        train_loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = loss_fn(y_val_pred, y_val)
        
        train_losses.append(train_loss.item())
        val_losses.append(val_loss.item())
        
        if (epoch+1) % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Val Loss: {val_loss.item():.4f}")
    
    return model, train_losses, val_losses

# --- Evaluation function on test set ---
def evaluate_on_test(model, X_test, y_test):
    
    model.eval()
    with torch.no_grad():
        y_test_pred = model(X_test)
        mse_test = mean_squared_error(y_test.numpy(), y_test_pred.numpy())

    print(f"\nFinal Test MSE (log scale): {mse_test:.4f}")
    return y_test_pred, mse_test

Note that the log-MSE is effectively the same across three optimizer choices: SGD, Adam, and RMSprop. 

In SGD, only one training example is used to compute the gradient and update the parameters at each iteration. This can be faster than batch gradient descent but may lead to more noise in the updates.

In RMSprop the learning rate is adaptively adjusted for each parameter based on the moving average of the squared gradient. This helps the algorithm to converge faster in the presence of noisy gradients.

Adam stands for adaptive moment estimation, it combines the benefits of Momentum-based Gradient Descent, Adagrad, and RMSprop the learning rate is adaptively adjusted for each parameter based on the moving average of the gradient and the squared gradient, which allows for faster convergence and better performance on non-convex optimization problems. It keeps track of two exponentially decaying averages the first-moment estimate, which is the exponentially decaying average of past gradients, and the second-moment estimate, which is the exponentially decaying average of past squared gradients. The first-moment estimate is used to calculate the momentum, and the second-moment estimate is used to scale the learning rate for each parameter. This is one of the most popular optimization algorithms for deep learning.

I elect to use the Adam optimizer. 

In [8]:
torch.manual_seed(42)
model = Perceptron(X_train_t.shape[1]); print(model)

# Compare optimizers and store test predictions / MSE
loss_fn = nn.MSELoss()

optimizers = [
    ("SGD", torch.optim.SGD),
    ("Adam", torch.optim.Adam),
    ("RMSprop", torch.optim.RMSprop),
]

y_test_preds = {}
mse_tests = {}

for name, opt_cls in optimizers:
    model = Perceptron(X_train_t.shape[1]) # new  model each run
    optimizer = opt_cls(model.parameters(), lr=0.01)
    
    model, train_losses, val_losses = train_slp(
        model, loss_fn, optimizer,
        X_train_t, y_train_t,
        X_val_t, y_val_t,
        epochs=2500
    )
    
    y_test_pred, mse_test = evaluate_on_test(model, X_test_t, y_test_t)
    y_test_preds[name] = y_test_pred.clone().detach()
    mse_tests[name] = mse_test

print("MSE per optimizer:", mse_tests)

Perceptron(
  (linear): Linear(in_features=10, out_features=1, bias=True)
)
Epoch 100/2500, Train Loss: 2.4170, Val Loss: 2.4088
Epoch 200/2500, Train Loss: 1.5779, Val Loss: 1.6070
Epoch 300/2500, Train Loss: 1.5627, Val Loss: 1.5930
Epoch 400/2500, Train Loss: 1.5623, Val Loss: 1.5927
Epoch 500/2500, Train Loss: 1.5622, Val Loss: 1.5926
Epoch 600/2500, Train Loss: 1.5621, Val Loss: 1.5926
Epoch 700/2500, Train Loss: 1.5621, Val Loss: 1.5925
Epoch 800/2500, Train Loss: 1.5620, Val Loss: 1.5924
Epoch 900/2500, Train Loss: 1.5620, Val Loss: 1.5924
Epoch 1000/2500, Train Loss: 1.5619, Val Loss: 1.5923
Epoch 1100/2500, Train Loss: 1.5619, Val Loss: 1.5923
Epoch 1200/2500, Train Loss: 1.5618, Val Loss: 1.5922
Epoch 1300/2500, Train Loss: 1.5618, Val Loss: 1.5922
Epoch 1400/2500, Train Loss: 1.5617, Val Loss: 1.5921
Epoch 1500/2500, Train Loss: 1.5617, Val Loss: 1.5921
Epoch 1600/2500, Train Loss: 1.5617, Val Loss: 1.5921
Epoch 1700/2500, Train Loss: 1.5616, Val Loss: 1.5920
Epoch 1800/2500

#### Early Stopping
- early stopping 
    - decompose into training, validation, and test data 
    - stop if: (test loss decreases) or (validation loss stops decreasing up to some threshold)

In [9]:
class EarlyStopping:
    """Track validation loss and stop training when it fails to improve.

    Args:
        patience (int): Number of consecutive non-improving epochs allowed.
        min_delta (float): Minimum change to qualify as an improvement.
        mode (str): 'min' to minimize loss, 'max' to maximize metric.

    Attributes:
        best (float): Best observed value so far.
        num_bad_epochs (int): Count of consecutive non-improving epochs.
        best_state (dict|None): State dict of the model at best value.
    """
    def __init__(self, patience=10, min_delta=1e-4, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        # Initialize best according to optimization direction
        self.best = math.inf if mode == 'min' else -math.inf
        self.num_bad_epochs = 0
        self.best_state = None

    def step(self, value, model):
        # Determine if current value is a meaningful improvement
        if self.mode == 'min':
            improved = (self.best - value) > self.min_delta
        else:
            improved = (value - self.best) > self.min_delta

        if improved:
            # Reset bad epoch counter, save best value and weights
            self.best = value
            self.num_bad_epochs = 0
            # Save the best model state for potential later use; deepcopy to avoid reference issues
            self.best_state = deepcopy(model.state_dict()) 
            return False  # continue training
        else:
            # Increment bad epochs and decide whether to stop
            self.num_bad_epochs += 1
            return self.num_bad_epochs >= self.patience  # stop if patience exceeded

Now I re-define my training procedure with the early-stopping. 

In [10]:
def train_slp(model, loss_fn, optimizer,
              X_train, y_train,
              X_val, y_val,
              max_epochs=2500, patience=10, min_delta=1e-4, trace_every=100):
    
    train_losses, val_losses = [], []
    es = EarlyStopping(patience=patience, min_delta=min_delta, mode='min')

    for epoch in range(1, max_epochs + 1):
        # train (full-batch)
        model.train()
        optimizer.zero_grad()
        y_train_pred = model(X_train)
        train_loss = loss_fn(y_train_pred, y_train)
        train_loss.backward()
        optimizer.step()

        # validate
        model.eval()
        with torch.no_grad():
            y_val_pred = model(X_val)
            val_loss = loss_fn(y_val_pred, y_val)

        train_losses.append(train_loss.item())
        val_losses.append(val_loss.item())

        if (epoch % trace_every) == 0 or epoch == 1:
            print(f"Epoch {epoch}/{max_epochs} - train {train_losses[-1]:.4f} - val {val_losses[-1]:.4f}")

        # early stopping checks on validation loss
        if es.step(val_losses[-1], model):
            print(f"Early stopping at epoch {epoch} (best val={es.best:.4f})")
            break

    # restore best weights
    if es.best_state is not None:
        model.load_state_dict(es.best_state)

    return model, train_losses, val_losses

In [12]:
#Example usage with your loaders
torch.manual_seed(42)
model = Perceptron(X_train_t.shape[1])
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


# Train model
model, train_losses, val_losses = train_slp(model, loss_fn, optimizer, 
                                              X_train_t, y_train_t, 
                                              X_val_t, y_val_t, 
                                              patience = 10, min_delta = 1e-4,
                                              max_epochs=2500)

# Evaluate final model on test set 
y_test_pred_t, mse_test = evaluate_on_test(model, X_test_t, y_test_t)

Epoch 1/2500 - train 44.1790 - val 44.0884
Epoch 100/2500 - train 32.2262 - val 32.1562
Epoch 200/2500 - train 23.1218 - val 23.0757
Epoch 300/2500 - train 16.2599 - val 16.2337
Epoch 400/2500 - train 11.2444 - val 11.2344
Epoch 500/2500 - train 7.7038 - val 7.7064
Epoch 600/2500 - train 5.2999 - val 5.3120
Epoch 700/2500 - train 3.7370 - val 3.7559
Epoch 800/2500 - train 2.7678 - val 2.7914
Epoch 900/2500 - train 2.1968 - val 2.2234
Epoch 1000/2500 - train 1.8783 - val 1.9067
Epoch 1100/2500 - train 1.7106 - val 1.7400
Epoch 1200/2500 - train 1.6275 - val 1.6574
Epoch 1300/2500 - train 1.5888 - val 1.6190
Epoch 1400/2500 - train 1.5719 - val 1.6023
Epoch 1500/2500 - train 1.5651 - val 1.5954
Epoch 1600/2500 - train 1.5625 - val 1.5928
Early stopping at epoch 1652 (best val=1.5923)

Final Test MSE (log scale): 1.5664


In [ ]:
# --- Plot training and validation loss
epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss", linestyle='dashed')
plt.xlabel("Epoch", fontsize=14)
plt.ylabel("MSE Loss", fontsize=14)
plt.title("MLP Training & Validation Loss", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

#### 